# Process Discovery Quality — Fitness, Precision, Generalization, Simplicity

A copy of `results_process` restructured around the four **classic process-discovery
dimensions**, followed by the **time metrics** (activity duration and case span).

Column order:
1. **Fitness · Precision · Generalization · Simplicity** — quality of the *discovered
   model* (higher = better).
2. **Evt-Ratio Err · Dur WAPE · Dur MAE · Span WAPE · Span MAE** — timing errors of the *simulation*
   (lower = better).

**Edge-F1 is deliberately not shown.**

Same row structure as `results_process`: three **process methods** × three
**time-approach** methods.

- **Alpha** — the alpha-miner net (naive process-discovery baseline).
- **Combined-best** — best discovered model per process (selection unchanged from
  `results_process`, so the two notebooks agree).
- **Budget** — best model + total-duration budgeting (`petri_net_budget`).
- Time approaches: **baseline** · **ml_global** (`_ml_plus_global`) ·
  **ml_local** (`_ml_plus_per_act`).

> **Read the first four columns as model-level, not simulation-level.** They are
> computed by replaying the **real** log against the discovered net
> (`modelling.py`, section 6), per station and averaged — Simplicity looks only at
> the net. They therefore do **not** depend on the simulation policy, so
> Combined-best and Budget (which share the same heuristic net) and all three
> time-prediction variants report *identical* values. Only the Alpha row differs.
> The time columns are where the nine rows actually separate.

Aggregation across processes = **median**. `wip_aware` / `wip_branching_aware`
are excluded.

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
EXPERIMENT = 963
SPLIT      = 'test'
AGG        = 'median'      # aggregation across processes for the combined table

PROCESS_METHODS = {
    'Alpha':         True,
    'Combined-best': True,
    'Budget':        True,
}
TIME_PREDICTIONS = {
    'baseline':  True,
    'ml_global': True,
    'ml_local':  True,
}
DISABLED_MINERS = ['wip_aware', 'wip_branching_aware']

# Combined-best selection metric — kept identical to `results_process` on purpose,
# so both notebooks always resolve to the same model. These are exactly the four
# columns this notebook displays first.
# Model selection ALWAYS happens on TRAIN, independently of SPLIT (which only
# controls what the tables report). Selecting on the split being reported would
# let a model be chosen for fitting the evaluation data.
SELECTION_SPLIT = 'train'

SELECTION_METRIC_BASES = [
    'conformance_metrics_fitness',
    'conformance_metrics_precision',
    'conformance_metrics_generalization',
    'conformance_metrics_simplicity',
]
# The four selection metrics are QUALITY scores (higher = better), so the best
# model is the argmax of their average -- not the argmin used when the criterion
# was expressed as errors.
SELECTION_HIGHER_IS_BETTER = True
SHOW_EVENT_RATIO = True    # Evt-Ratio Err sits after Simplicity, before the time metrics
SAVE_LATEX     = True
COMPACT_LATEX  = True      # \scriptsize + tight tabcolsep + 2-line row labels/headers

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
sp = SPLIT.lower() + '_'
assert any(c.startswith(sp) for c in df.columns), f'No {sp}* columns found'

def parse_mode(m):
    r = str(m)
    if not r.startswith('petri_net_'):
        return r, 'baseline'
    r = r[len('petri_net_'):]
    if r.endswith('_ml_plus_global'):
        return r[:-len('_ml_plus_global')], 'ml_global'
    if r.endswith('_ml_plus_per_act'):
        return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'

df['model'], df['time_pred'] = zip(*df['mode'].map(parse_mode))
df = df[~df['model'].isin(DISABLED_MINERS)].copy()
print(f'{len(df)} rows | processes: {sorted(df["process"].unique())} | '
      f'models: {sorted(df["model"].unique())}')

Loading: experiment_963_20260722_180634
54 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5'] | models: ['alpha', 'budget', 'heuristic']


In [3]:
# ── Metrics: discovery quality FIRST (higher = better), then timing (lower) ──
# Unlike `results_process` (all-errors), this table mixes directions, so every
# metric carries an explicit 'max'/'min' so highlighting and LaTeX bolding pick
# the right end.
_wape_ok   = (sp + 'duration_metrics_activity_duration_wape') in df.columns
_dur_base  = ('duration_metrics_activity_duration_wape' if _wape_ok
              else 'duration_metrics_activity_duration_error')
_span_base = ('duration_metrics_case_span_wape'
              if (sp + 'duration_metrics_case_span_wape') in df.columns
              else 'duration_metrics_case_span_error')

# Columns: (parquet base, display label, LaTeX header, decimals, direction)
# ── Change column headers / decimals HERE ───────────────────────────────────
METRICS = [
    ('conformance_metrics_fitness',           'Fitness',        r'Fitness',                      3, 'max'),
    ('conformance_metrics_precision',         'Precision',      r'Precision',                    3, 'max'),
    ('conformance_metrics_generalization',    'Generalization', r'\makecell{General-\\ization}', 3, 'max'),
    ('conformance_metrics_simplicity',        'Simplicity',     r'Simplicity',                   3, 'max'),
]
if SHOW_EVENT_RATIO:
    METRICS.append(
        ('basic_metrics_event_count_error',   'Evt-Ratio Err',  r'\makecell{Evt-Ratio\\Err}',    3, 'min'))
METRICS += [
    (_dur_base,                               'Dur WAPE (%)',   r'\makecell{Dur\\WAPE (\%)}',    1, 'min'),
    ('duration_metrics_activity_duration_mae','Dur MAE (min)',  r'\makecell{Dur MAE\\(min)}',    1, 'min'),
    (_span_base,                              'Span WAPE (%)',  r'\makecell{Span\\WAPE (\%)}',   1, 'min'),
    ('duration_metrics_case_span_mae',        'Span MAE (min)', r'\makecell{Span MAE\\(min)}',   1, 'min'),
]

METRICS   = [m for m in METRICS if (sp + m[0]) in df.columns]
COL_OF    = {lbl: sp + b for (b, lbl, tex, dp, d) in METRICS}
LABELS    = [lbl for (b, lbl, tex, dp, d) in METRICS]
TEX_HDR   = {lbl: tex for (b, lbl, tex, dp, d) in METRICS}
DECIMALS  = {lbl: dp  for (b, lbl, tex, dp, d) in METRICS}   # used by HTML *and* LaTeX
DIRECTION = {lbl: d   for (b, lbl, tex, dp, d) in METRICS}

def best_of(series, lbl):
    # Direction-aware best value of a column slice; None when nothing is finite.
    v = series.dropna()
    if v.empty:
        return None
    return v.max() if DIRECTION[lbl] == 'max' else v.min()

print('Discovery :', [l for l in LABELS if DIRECTION[l] == 'max'])
print('Timing    :', [l for l in LABELS if DIRECTION[l] == 'min'])

Discovery : ['Fitness', 'Precision', 'Generalization', 'Simplicity']
Timing    : ['Evt-Ratio Err', 'Dur WAPE (%)', 'Dur MAE (min)', 'Span WAPE (%)', 'Span MAE (min)']


In [4]:
# ── Combined-best process model per process (identical rule to results_process) ─
_sel_prefix = SELECTION_SPLIT.lower() + '_'          # selection on TRAIN
_sel_cols = [_sel_prefix + b for b in SELECTION_METRIC_BASES
             if (_sel_prefix + b) in df.columns]
assert _sel_cols, f'no {_sel_prefix}* selection columns found'
_candidates = sorted(set(df['model'].unique()) - {'alpha', 'budget'} - set(DISABLED_MINERS))

best_miner = {}
for proc, g in df.groupby('process'):
    gc = g[g['model'].isin(_candidates)]
    if gc.empty:
        best_miner[proc] = None
        continue
    score = gc.groupby('model')[_sel_cols].mean().mean(axis=1)   # avg of the 4 dimensions
    best_miner[proc] = score.idxmax() if SELECTION_HIGHER_IS_BETTER else score.idxmin()

print(f'Candidates for Combined-best (selected on {SELECTION_SPLIT.upper()}):', _candidates)
for p, m in best_miner.items():
    print(f'  {p}: {m}')

def model_for(proc, pm_label):
    if pm_label == 'Alpha':         return 'alpha'
    if pm_label == 'Budget':        return 'budget'
    if pm_label == 'Combined-best': return best_miner.get(proc)
    return None

Candidates for Combined-best (selected on TRAIN): ['heuristic']
  process_1: heuristic
  process_2: heuristic
  process_3: heuristic
  process_4_1: heuristic
  process_4_2: heuristic
  process_5: heuristic


In [5]:
# ── Assemble long table then aggregate across processes ──────────────────────
PM_ORDER   = [k for k in ['Alpha', 'Combined-best', 'Budget'] if PROCESS_METHODS.get(k)]
TIME_ORDER = [k for k in ['baseline', 'ml_global', 'ml_local'] if TIME_PREDICTIONS.get(k)]

records = []
for proc, g in df.groupby('process'):
    for pm in PM_ORDER:
        mdl = model_for(proc, pm)
        if mdl is None:
            continue
        for _, row in g[g['model'] == mdl].iterrows():
            if row['time_pred'] not in TIME_ORDER:
                continue
            rec = {'process': proc, 'Process method': pm, 'Time approach': row['time_pred']}
            for lbl, col in COL_OF.items():
                rec[lbl] = row[col]
            records.append(rec)

long_df = pd.DataFrame(records)

def build_combined(long_df, agg='median'):
    t = long_df.groupby(['Process method', 'Time approach'])[LABELS].agg(agg)
    idx = pd.MultiIndex.from_tuples(
        [(pm, tp) for pm in PM_ORDER for tp in TIME_ORDER if (pm, tp) in t.index],
        names=['Process method', 'Time approach'])
    return t.reindex(idx)

combined = build_combined(long_df, agg=AGG)

def _fmt(v, lbl):
    return '' if pd.isna(v) else f'{v:.{DECIMALS[lbl]}f}'

def style_table(tbl):
    best = {c: best_of(tbl[c], c) for c in tbl.columns}
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for col in tbl.columns:
            b = best[col]
            if b is None:
                continue
            for idx in tbl.index:
                v = tbl.loc[idx, col]
                if pd.notna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    fmt = {lbl: (lambda v, l=lbl: _fmt(v, l)) for lbl in tbl.columns}
    return (tbl.style.format(fmt).apply(_styler, axis=None)
              .set_caption(f'Discovery quality (higher = better) + timing errors '
                           f'(lower = better) — {AGG} across processes'))

display(Markdown(f'### Discovery quality + timing — all processes ({AGG})'))
display(style_table(combined))

### Discovery quality + timing — all processes (median)

## Per process — one large combined table

Every process shown **individually**. Best value per column **within each process**
is highlighted / bolded — direction-aware (max for the four discovery dimensions,
min for the timing errors).

In [6]:
# ── Large per-process table: MultiIndex (Process, method, time prediction) ────
def build_per_process(long_df):
    t = long_df.groupby(['process', 'Process method', 'Time approach'])[LABELS].median()
    procs = sorted(long_df['process'].unique())
    idx = pd.MultiIndex.from_tuples(
        [(p, pm, tp) for p in procs for pm in PM_ORDER for tp in TIME_ORDER
         if (p, pm, tp) in t.index],
        names=['Process', 'Process method', 'Time approach'])
    return t.reindex(idx)

per_process = build_per_process(long_df)

def _best_within_process(tbl):
    best = {}
    for proc, sub in tbl.groupby(level=0):
        for col in tbl.columns:
            best[(proc, col)] = best_of(sub[col], col)
    return best

def style_per_process(tbl):
    best = _best_within_process(tbl)
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for idx in tbl.index:
            for col in tbl.columns:
                v = tbl.loc[idx, col]
                b = best.get((idx[0], col))
                if b is not None and pd.notna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    fmt = {lbl: (lambda v, l=lbl: _fmt(v, l)) for lbl in tbl.columns}
    return (tbl.style.format(fmt).apply(_styler, axis=None)
              .set_caption('Per-process — best per column within each process '
                           '(discovery: higher; timing: lower)'))

display(Markdown('### Discovery quality + timing — per process (large table)'))
display(style_per_process(per_process))

### Discovery quality + timing — per process (large table)

## Discovery dimensions alone — Alpha vs the discovered model

Because Fitness / Precision / Generalization / Simplicity are model-level, the nine
rows above collapse to **two distinct values per process**. This compact table drops
the redundant rows and shows the discovery comparison on its own — one row per
(process, miner).

In [7]:
DISC = [l for l in LABELS if DIRECTION[l] == 'max']
_disc_rows = (df[df['model'].isin(['alpha'] + _candidates)]
                .groupby(['process', 'model'])[[COL_OF[l] for l in DISC]].first())
_disc_rows.columns = DISC
_disc_rows.index.names = ['Process', 'Miner']

def style_disc(tbl):
    best = {}
    for proc, sub in tbl.groupby(level=0):
        for col in tbl.columns:
            best[(proc, col)] = best_of(sub[col], col)
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for idx in tbl.index:
            for col in tbl.columns:
                v = tbl.loc[idx, col]; b = best.get((idx[0], col))
                if b is not None and pd.notna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    return (tbl.style.format({c: '{:.3f}' for c in tbl.columns}, na_rep='—')
              .apply(_styler, axis=None)
              .set_caption('Process discovery quality — higher = better'))

display(style_disc(_disc_rows))
display(Markdown(f'**Mean across processes**'))
display(_disc_rows.groupby('Miner').mean().round(3))

**Mean across processes**

,Fitness,Precision,Generalization,Simplicity
Miner,,,,
alpha,0.633,0.456,0.606,0.489
heuristic,0.990,0.748,0.678,0.676


## LaTeX

MultiIndex rows, best value per column bolded (globally for the aggregated table,
within each process for the per-process tables), direction-aware. Preamble needs
`\usepackage{makecell}` and `\usepackage{booktabs}`.

In [8]:
# ── LaTeX layout knobs — change these if the table doesn't fit ──────────────
LATEX_FONT        = r'\footnotesize'  # \small > \footnotesize > \scriptsize > \tiny
LATEX_TABCOLSEP   = '4pt'             # column padding (LaTeX default 6pt); lower = narrower
LATEX_STRETCH     = '1.0'             # row height multiplier; lower = shorter rows
SHORT_PROCESS_IDS = True              # process_4_1 -> P4.1 in the per-process table

# Paper table (aggregated) knobs — the two-column `table*` float used in the paper.
PAPER_STRETCH = '1.3'
PAPER_FONT    = r'\footnotesize'
# Row labels of the aggregated paper table (level-0 index -> LaTeX).
PAPER_ROW = {
    'Alpha':         r'\makecell[c]{Alpha \\ (Baseline)}',
    'Combined-best': r'\makecell[c]{Best \\ Petri Net}',
    'Budget':        r'\makecell[c]{Best \\ Petri Net \\ + Budget}',
}
# Decimals used ONLY by the paper table. Substring rules first (so a renamed
# column still matches), then exact-label overrides, then DECIMALS.
PAPER_DECIMALS_RULES = [('WAPE', 3)]      # any column whose label contains 'WAPE'
PAPER_DECIMALS       = {}                 # exact-label overrides, e.g. {'Simplicity': 2}
# Column headers of the aggregated paper table (falls back to TEX_HDR).
PAPER_HDR = {
    'Fitness':        r'Fitness',
    'Precision':      r'Precision',
    'Generalization': r'\makecell{Generalization}',
    'Simplicity':     r'Simplicity',
    'Evt-Ratio Err':  r'\makecell{Evt-Ratio\\Error}',
    'Dur WAPE (%)':   r'\makecell{Dur-act\\WAPE (\%)}',
    'Dur MAE (min)':  r'\makecell{Dur-act\\MAE (min)}',
    'Span WAPE (%)':  r'\makecell{Case \\span\\WAPE (\%)}',
    'Span MAE (min)': r'\makecell{Case \\span MAE\\(min)}',
}

# Only labels that genuinely need it are split over two lines.
TEX_ROW = {
    'Combined-best': r'\makecell[l]{Combined-\\best}',
}
# Index level names as they appear in the LaTeX header row.
TEX_IDX_NAME = {'Process method': 'Method', 'Time approach': r'\makecell[l]{Time\\approach}'}

def _tex_esc(s):
    return str(s).replace('_', r'\_')

def _tex_proc(p):
    # process_4_1 -> P4.1 ; process_2 -> P2
    if not SHORT_PROCESS_IDS:
        return str(p)
    return 'P' + str(p).replace('process_', '').replace('_', '.')

def _tex_idx(tbl):
    # Map index values through TEX_ROW / _tex_proc and rename the level headers.
    lv = []
    for i, name in enumerate(tbl.index.names):
        vals = tbl.index.get_level_values(i)
        lv.append([_tex_proc(v) for v in vals] if name == 'Process'
                  else [TEX_ROW.get(v, str(v)) for v in vals])
    out = tbl.copy()
    out.index = pd.MultiIndex.from_arrays(
        lv, names=[TEX_IDX_NAME.get(n, n) for n in tbl.index.names])
    return out

def to_latex_table(tbl, caption, label, best_map=None, n_index=2):
    # best_map: dict[(group_key, col)] -> best value, group_key = row-index level 0.
    # None -> best computed globally per column. Direction comes from DIRECTION.
    if best_map is None:
        best_map = {('__global__', c): best_of(tbl[c], c) for c in tbl.columns}
        key_of = lambda idx: '__global__'
    else:
        key_of = lambda idx: idx[0]
    str_df = pd.DataFrame(index=tbl.index, columns=tbl.columns, dtype=object)
    for col in tbl.columns:
        for idx in tbl.index:
            v = tbl.loc[idx, col]
            if pd.isna(v):
                str_df.loc[idx, col] = ''
            else:
                b = best_map.get((key_of(idx), col))
                s = _fmt(v, col)
                str_df.loc[idx, col] = ((r'\textbf{' + s + '}')
                                        if (b is not None and abs(v - b) < 1e-9) else s)
    str_df = _tex_idx(str_df)
    str_df.columns = [TEX_HDR[c] for c in tbl.columns]
    latex = str_df.to_latex(
        escape=False, multirow=True,
        column_format=('l' * n_index) + '|' + '|'.join(['c'] * len(tbl.columns)),
        caption=caption, label=label, position='H',
    )
    if COMPACT_LATEX:
        latex = latex.replace(
            r'\begin{tabular}',
            f'\\centering\n\\setlength{{\\tabcolsep}}{{{LATEX_TABCOLSEP}}}\n'
            f'\\renewcommand{{\\arraystretch}}{{{LATEX_STRETCH}}}\n{LATEX_FONT}\n'
            r'\begin{tabular}', 1)
    return latex.replace('_', r'\_')

def _paper_dp(lbl):
    # Decimals for one column of the paper table.
    if lbl in PAPER_DECIMALS:
        return PAPER_DECIMALS[lbl]
    for pat, dp in PAPER_DECIMALS_RULES:
        if pat.lower() in str(lbl).lower():
            return dp
    return DECIMALS[lbl]

def _fmt_paper(v, lbl):
    # Same as _fmt but honours PAPER_DECIMALS_RULES / PAPER_DECIMALS.
    return '' if pd.isna(v) else f'{v:.{_paper_dp(lbl)}f}'

def to_paper_table(tbl, caption, label, description):
    # Hand-rolled `table*` float in the exact paper layout: caption on top,
    # booktabs rules, one \multirow block per process method separated by \cline,
    # and a \footnotesize description paragraph under the tabular.
    # Bolding is global per column and direction-aware (DIRECTION).
    n_idx = tbl.index.nlevels
    ncol  = n_idx + len(tbl.columns)
    best  = {c: best_of(tbl[c], c) for c in tbl.columns}

    def cell(v, col):
        if pd.isna(v):
            return ''
        s, b = _fmt_paper(v, col), best[col]
        return (r'\textbf{' + s + '}') if (b is not None and abs(v - b) < 1e-9) else s

    hdr = ([TEX_IDX_NAME.get(n, str(n)) for n in tbl.index.names]
           + [PAPER_HDR.get(c, TEX_HDR[c]) for c in tbl.columns])

    L = [r'\begin{table*}[t]',
         r'\centering',
         rf'\caption{{{caption}}}',
         rf'\label{{{label}}}',
         r'\vspace{-0.5em}',
         rf'\setlength{{\tabcolsep}}{{{LATEX_TABCOLSEP}}}',
         rf'\renewcommand{{\arraystretch}}{{{PAPER_STRETCH}}}',
         PAPER_FONT,
         r'\begin{tabular}{' + 'l' * n_idx + '|'
             + '|'.join(['c'] * len(tbl.columns)) + '}',
         r'\toprule',
         ' & '.join(hdr) + r' \\',
         r'\midrule']

    for pm in dict.fromkeys(tbl.index.get_level_values(0)):      # keeps PM_ORDER
        sub = tbl.xs(pm, level=0, drop_level=False)
        head = rf'\multirow{{{len(sub)}}}{{*}}{{{PAPER_ROW.get(pm, _tex_esc(pm))}}}'
        for idx, row in sub.iterrows():
            L.append(' & '.join([head, _tex_esc(idx[-1])]
                                + [cell(row[c], c) for c in tbl.columns]) + r' \\')
            head = ''                                            # only the first row
        L.append(rf'\cline{{1-{ncol}}}')

    L += [r'\bottomrule',
          r'\end{tabular}',
          r'\vspace{0.5em}',
          r'\noindent\raggedright\footnotesize',
          '',
          description,
          '',
          r'\end{table*}']
    return '\n'.join(L)

# LaTeX is printed only — no process_metrics_tables/ folder is written.

_dir_note = (r'Fitness, Precision, Generalization and Simplicity are discovery quality '
             r'(higher is better); Evt-Ratio Err and the Dur/Span columns are errors '
             r'(lower is better). \textbf{Bold} = best per column')

# ── Aggregated table, paper layout ──────────────────────────────────────────
_paper_desc = (
    r'Description: Process discovery methods are Alpha (naive miner baseline), '
    r'Best Petri Net (best discovered Petri Net), Best Petri Net + Budget (best Petri '
    r'Net + duration budgeting at simulation time), each crossed with the three '
    r'activity-duration predictors (baseline is the median, ml\_local is a ML model '
    r'per activity and ml\_global is a ML model for all activities). Fitness, '
    r'Precision, Generalization and Simplicity are discovery quality (higher is '
    r'better); Evt-Ratio Error is the relation between total simulated events/number '
    r'of real events, lower is better; and the Dur-act are the difference of the '
    r'individual activities durations vs the observed in test set and Span are the '
    r'same but for the trace length (lower is better). \textbf{Bold} = best per column.')

print('paper-table decimals:', {c: _paper_dp(c) for c in combined.columns})

tex_agg = to_paper_table(
    combined,
    caption=(f'Process modelling and timing accuracy ({SPLIT} set results, {AGG} '
             r'across processes and cases)'),
    label=_tex_esc(f'tab:process_discovery_{EXPERIMENT}'),
    description=_paper_desc)

tex_pp = to_latex_table(
    per_process,
    caption=(f'Process discovery quality and timing accuracy per process ({SPLIT} set). '
             r'Same structure as Table~\ref{tab:process_discovery_' + str(EXPERIMENT) +
             r'} but every process shown individually. ' + _dir_note + ' within each process.'),
    label=f'tab:process_discovery_perproc_{EXPERIMENT}',
    best_map=_best_within_process(per_process), n_index=3)

_disc_tbl = _disc_rows.copy()
tex_disc = to_latex_table(
    _disc_tbl,
    caption=(f'Process discovery quality per process ({SPLIT} set): Fitness, Precision, '
             r'Generalization and Simplicity of the discovered Petri net, obtained by '
             r'replaying the real log against the net (per station, averaged). Higher is '
             r'better; \textbf{bold} = best miner per metric within each process.'),
    label=f'tab:process_discovery_only_{EXPERIMENT}',
    best_map={(p, c): best_of(s[c], c) for p, s in _disc_tbl.groupby(level=0)
              for c in _disc_tbl.columns},
    n_index=2)


print('% ===== AGGREGATED =====');       print(tex_agg)
print('\n% ===== PER PROCESS =====');    print(tex_pp)
print('\n% ===== DISCOVERY ONLY ====='); print(tex_disc)


paper-table decimals: {'Fitness': 3, 'Precision': 3, 'Generalization': 3, 'Simplicity': 3, 'Evt-Ratio Err': 3, 'Dur WAPE (%)': 3, 'Dur MAE (min)': 1, 'Span WAPE (%)': 3, 'Span MAE (min)': 1}
% ===== AGGREGATED =====
\begin{table*}[t]
\centering
\caption{Process modelling and timing accuracy (test set results, median across processes and cases)}
\label{tab:process\_discovery\_963}
\vspace{-0.5em}
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.3}
\footnotesize
\begin{tabular}{ll|c|c|c|c|c|c|c|c|c}
\toprule
Method & \makecell[l]{Time\\approach} & Fitness & Precision & \makecell{Generalization} & Simplicity & \makecell{Evt-Ratio\\Error} & \makecell{Dur-act\\WAPE (\%)} & \makecell{Dur-act\\MAE (min)} & \makecell{Case \\span\\WAPE (\%)} & \makecell{Case \\span MAE\\(min)} \\
\midrule
\multirow{3}{*}{\makecell[c]{Alpha \\ (Baseline)}} & baseline & 0.635 & 0.420 & 0.556 & 0.477 & 0.496 & 49.869 & 6.6 & 70.341 & 337.5 \\
 & ml\_global & 0.635 & 0.420 & 0.556 & 0.477 & 0.496 & 66.230